# Moment 1: discouraged_share

StatsSA's discouraged work-seeker rate, from QLFS microdata. Matches `src/moments.py`'s
`discouraged_share()` in units: a share of the **expanded labour force** (employed +
unemployed-narrow + discouraged), not of the full working-age population. The model's own
`n_agents` never includes agents who are structurally outside the labour force by
construction (there's no scholar/retiree/home-maker agent type) -- every agent is either
working, actively searching, or discouraged from searching, which is exactly what StatsSA's
expanded labour force concept describes, and exactly the same convention StatsSA itself uses
when it publishes "discouraged work-seekers as a percentage of the expanded labour force."
Using the *full* working-age population as the denominator instead would dilute the share with
people the model never represents (scholars, retirees, home-makers), and wasn't used.

**Source variable**: QLFS's own pre-built `Status` classification (`Employed` /
`Unemployed` / `Discouraged job seeker` / `Other not economically active`), not reconstructed
from the underlying reason codes (`Q38RSNNOTSEEK`, `InactReason`) -- StatsSA has already done
that classification, and re-deriving it independently would just be a second, less reliable
copy of the same thing with more room for error.

**Quarters used**: the four most recent with clean microdata on disk -- 2025 Q2, Q3, Q4 and
2026 Q1 (2025 Q1 exists as `qlfs-2025-q1-worker-v1` but was excluded to keep a clean rolling
12-month window ending at the most recent release, rather than a window with an extra quarter
tacked on one end).

**Weighting**: QLFS's own person-level `Weight` variable. The standard error uses a Kish
approximate design effect from the weight distribution (`deff = n * sum(w^2) / sum(w)^2`),
not a full Taylor-linearised or replicate-weight survey design -- `Stratum` exists in the file
but no PSU/cluster variable was found, so clustering isn't accounted for and the reported SE is
a reasonable approximation, understating the true design-based SE somewhat rather than
overstating it. Documented here rather than silently treated as exact -- see
`paper/notes/prompt-0.2-licence-check.md`'s sibling note on what wasn't done for the licence
check; the same honesty standard applies to the statistics themselves.

**Headline value**: the most recent quarter (2026 Q1) alone, not a four-quarter pool. There's a
real, non-trivial upward trend across the four quarters -- see the by-quarter table below -- and
pooling would average over it rather than report it, understating how far the current value has
moved. `moments.csv`'s schema expects one `period` per moment; the by-quarter series is kept in
this notebook as context, not folded into a single blended number.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings(
    "ignore", category=UnicodeWarning
)  # QLFS string fields fall back to latin-1; expected, not a bug

# Edit this to wherever you extracted the DataFirst downloads -- see data/README.md.
DATA_ROOT = Path(r"C:\Users\aakas\Documents\Projects\Thesis\Data\extracted")

QUARTERS = {
    "2025-Q2": DATA_ROOT / "qlfs-2025-q2-v1" / "qlfs-2025-q2-v1.dta",
    "2025-Q3": DATA_ROOT / "qlfs-2025-03" / "QLFS202503.dta",
    "2025-Q4": DATA_ROOT / "qlfs-2025-04" / "qlfs-2025-q4-v1.dta",
    "2026-Q1": DATA_ROOT / "qlfs-2026-q1-v1" / "qlfs-2026-q1-v1.dta",
}

LABOUR_FORCE_CATEGORIES = ["Employed", "Unemployed", "Discouraged job seeker"]

frames = {}
for quarter, path in QUARTERS.items():
    df = pd.read_stata(path, columns=["Status", "Weight"], convert_categoricals=True)
    frames[quarter] = df[df["Status"].isin(LABOUR_FORCE_CATEGORIES)].copy()
    print(f"{quarter}: {len(frames[quarter]):,} expanded-labour-force respondents")

In [ ]:
def weighted_discouraged_share(df: pd.DataFrame) -> tuple[float, float, int, float]:
    """Weighted share of the expanded labour force that's discouraged, with a Kish
    approximate-design-effect standard error -- see the markdown cell above for exactly what
    this does and doesn't account for."""
    w = df["Weight"].to_numpy()
    is_discouraged = (df["Status"] == "Discouraged job seeker").to_numpy().astype(float)
    p_hat = np.average(is_discouraged, weights=w)
    n = len(df)
    deff = (w**2).sum() * n / (w.sum() ** 2)
    n_eff = n / deff
    se = np.sqrt(p_hat * (1 - p_hat) / n_eff)
    return p_hat, se, n, deff


results = []
for quarter, df in frames.items():
    p_hat, se, n, deff = weighted_discouraged_share(df)
    results.append({"quarter": quarter, "n": n, "discouraged_share": p_hat, "se": se, "deff": deff})

results_df = pd.DataFrame(results).set_index("quarter")
results_df

## Result

The by-quarter table shows a real upward trend (roughly 11.9% to 13.4% over four quarters, a
move several times larger than any single quarter's own standard error) -- worth carrying into
the Calibration chapter as a caveat, since the model's AR(1) shock mean-reverts and has no
mechanism for a secular trend like this. The headline moment is the most recent quarter,
2026 Q1, not a pooled average -- see the reasoning in the first cell.

In [ ]:
headline_quarter = "2026-Q1"
headline = results_df.loc[headline_quarter]

moments_path = Path("../../data/moments.csv")
moments = pd.read_csv(moments_path)
moments["period"] = moments["period"].astype("object")
moments["source"] = moments["source"].astype("object")
row = moments["key"] == "discouraged_share"
moments.loc[row, "value"] = round(float(headline["discouraged_share"]), 4)
moments.loc[row, "standard_error"] = round(float(headline["se"]), 4)
moments.loc[row, "period"] = headline_quarter
moments.loc[row, "source"] = (
    "Statistics South Africa. Quarterly Labour Force Survey 2026: Q1 [dataset]. "
    "Cape Town: DataFirst [distributor]. QLFS Status variable, expanded-labour-force "
    "denominator, person-weighted."
)
moments.loc[row, "provisional"] = False
moments.to_csv(moments_path, index=False)
moments[row]